# CSV Data Preprocessing for Training

This notebook converts `code_samples.csv` into the JSON format required by the training pipeline.

In [5]:
import pandas as pd
import json
from pathlib import Path
from sklearn.model_selection import train_test_split

# Set up paths
CSV_PATH = Path('code_samples.csv')
PROCESSED_DATA_DIR = Path('../data/processed')
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Environment ready!")

Environment ready!


In [6]:
CSV_PATH

WindowsPath('code_samples.csv')

## 1. Load CSV Data

In [7]:
df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df)} samples")
df.head()

Loaded 1000 samples


,instruction,response,language
0,Create EF Core DbSet for Sales,public DbSet<Sales> Saless { get; set; },.NET
1,Create Angular form for Category,"this.form = this.fb.group({ name: ['', Validat...",Angular
2,Write SQL query to get all Category,SELECT * FROM Category;,SQL
3,Create a C# class for Product,public class Product { public int Id { get; se...,.NET
4,Create a C# class for Purchase,public class Purchase { public int Id { get; s...,.NET


## 2. Map Columns and Add Context

The training pipeline expects headers: `language`, `framework`, `description`, `code`, `context`, `input`, `output`.

In [8]:
# Rename columns
df = df.rename(columns={
    'instruction': 'description',
    'response': 'code'
})

# Map languages to standard frameworks
framework_map = {
    '.NET': 'dotnet8',
    'Angular': 'angular',
    'SQL': 'mssql'
}

df['framework'] = df['language'].map(framework_map)
df['context'] = ''  # Default empty context

def create_training_pair(row):
    """Create input-output pair for training"""
    input_text = f"Language: {row['language']}\n"
    input_text += f"Framework: {row['framework']}\n"
    input_text += f"Task: {row['description']}\n"
    
    output_text = row['code']
    return input_text, output_text

df[['input', 'output']] = df.apply(
    lambda row: pd.Series(create_training_pair(row)), 
    axis=1
)

print("Data prepared!")
print(df.iloc[0]['input'])

Data prepared!
Language: .NET
Framework: dotnet8
Task: Create EF Core DbSet for Sales



## 3. Split and Save

In [9]:
train_df, val_df = train_test_split(df, test_size=0.3, random_state=42)

columns_to_save = ['language', 'framework', 'description', 'context', 'input', 'output']

train_df[columns_to_save].to_json(
    PROCESSED_DATA_DIR / 'train.json', 
    orient='records', 
    indent=2
)
val_df[columns_to_save].to_json(
    PROCESSED_DATA_DIR / 'validation.json', 
    orient='records', 
    indent=2
)

print(f"Saved {len(train_df)} training samples to {PROCESSED_DATA_DIR / 'train.json'}")
print(f"Saved {len(val_df)} validation samples to {PROCESSED_DATA_DIR / 'validation.json'}")

Saved 700 training samples to ..\data\processed\train.json
Saved 300 validation samples to ..\data\processed\validation.json
